# M3-B1 — Fusion de deux sources + colonne de provenance

> ~30-40 min. Le geste **exact** que le cas d'usage certif attend côté
> données : réunir **deux exports comparables** (même métier, deux
> origines) dans un seul tableau, **sans perdre d'où vient chaque ligne**.

Ici : deux sites d'Acerox t'envoient chacun un export de capteurs. Même
idée, mais ce **ne sont pas** exactement les mêmes fichiers. Ton travail :
les empiler proprement et repérer ce qui cloche.

In [31]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'data'
a = pd.read_csv(DATA_DIR / 'capteurs_site_A.csv')
b = pd.read_csv(DATA_DIR / 'capteurs_site_B.csv')
print('A', a.shape, '| colonnes', list(a.columns))
print('B', b.shape, '| colonnes', list(b.columns))

A (25, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'debit_l_min']
B (22, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'firmware']


## 1. Le réflexe naïf (à ne PAS garder)

On empile les deux sans réfléchir. Regarde bien le résultat.

In [32]:
naif = pd.concat([a, b], ignore_index=True)
naif.info()
display(naif.head())
display(naif.tail())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ts              47 non-null     object 
 1   machine_id      47 non-null     object 
 2   temp_c          47 non-null     float64
 3   vibration_mm_s  47 non-null     float64
 4   debit_l_min     25 non-null     float64
 5   firmware        22 non-null     object 
dtypes: float64(3), object(3)
memory usage: 2.3+ KB


,ts,machine_id,temp_c,vibration_mm_s,debit_l_min,firmware
0,2026-06-01T06:00:00,M01,72.5,1.77,124.7,NaN
1,2026-06-01T07:00:00,M04,69.9,2.36,125.7,NaN
2,2026-06-01T08:00:00,M04,64.6,2.40,126.3,NaN
3,2026-06-01T09:00:00,M03,69.5,2.32,117.2,NaN
4,2026-06-01T10:00:00,M03,64.2,1.83,116.3,NaN


,ts,machine_id,temp_c,vibration_mm_s,debit_l_min,firmware
42,2026-06-01T23:00:00,M05,156.6,3.30,NaN,v1.2
43,2026-06-02T00:00:00,M04,155.2,3.76,NaN,v1.3
44,2026-06-02T01:00:00,M05,166.2,2.51,NaN,v1.2
45,2026-06-02T02:00:00,M08,153.3,1.81,NaN,v1.2
46,2026-06-02T03:00:00,M05,147.8,1.23,NaN,v1.3


### Ce qui vient de casser

* Après le `pd.concat`, il est impossible de savoir si une ligne vient du site A ou du site B, car aucune colonne de provenance n’a été conservée.
* La colonne `debit_l_min` contient 22 valeurs manquantes, car elle existe uniquement dans l’export du site A. Le site B fournit à la place une colonne `firmware`.
* La colonne `firmware` contient 25 valeurs manquantes pour la même raison : elle n’existe pas dans l’export du site A.
* La colonne `temp_c` mélange probablement deux unités différentes. Les valeurs du site A sont proches de 65–75, tandis que celles du site B sont proches de 145–166, ce qui suggère que le site B utilise des degrés Fahrenheit malgré le nom de colonne `temp_c`.
* Certains identifiants de machines peuvent également exister dans les deux sites. Sans provenance, un identifiant comme `M04` peut devenir ambigu.

Cette fusion naïve conserve les lignes, mais elle perd le contexte nécessaire pour interpréter correctement les données.


### Observation

La fusion naïve contient bien les 47 lignes, mais elle présente plusieurs
problèmes :

- aucune colonne ne permet de savoir si une ligne vient du site A ou du site B ;
- `debit_l_min` est manquant sur les 22 lignes du site B, car cette colonne
  n'existe pas dans son export ;
- `firmware` est manquant sur les 25 lignes du site A ;
- `temp_c` semble mélanger des températures en Celsius et en Fahrenheit ;
- certains identifiants de machines sont présents dans les deux sites.

## 2. Le bon geste : marquer la provenance AVANT de concaténer

Une colonne `source` = une **fiche d'identité** de chaque ligne. C'est la
base de la traçabilité (RGPD, réconciliation, filtrage par scénario ensuite).

In [33]:
a2 = a.assign(source='site_A')
b2 = b.assign(source='site_B')
fusion = pd.concat([a2, b2], ignore_index=True)
fusion['source'].value_counts()

source
site_A    25
site_B    22
Name: count, dtype: int64

## 3. Maintenant, la provenance te fait VOIR les pièges

Fusionner n'est pas empiler. Trois choses à débusquer avec la colonne `source`.

In [34]:
# Piège A — mêmes machine_id des deux côtés : le « M04 » de A est-il
# le même équipement que le « M04 » de B ? (indice : non)
communs = set(a['machine_id']) & set(b['machine_id'])
print('machine_id présents dans les DEUX sites :', sorted(communs))
# TODO — que faire ? préfixer par la source ? (ex. 'site_A::M04')

machine_id présents dans les DEUX sites : ['M04', 'M05']


# Création d'un identifiant unique à l'échelle des deux sites.


In [35]:
fusion["machine_key"] = (
    fusion["source"]
    + "::"
    + fusion["machine_id"]
)

display(
    fusion[
        ["source", "machine_id", "machine_key"]
    ]
    .drop_duplicates()
    .sort_values(["machine_id", "source"])
)

,source,machine_id,machine_key
0,site_A,M01,site_A::M01
8,site_A,M02,site_A::M02
3,site_A,M03,site_A::M03
1,site_A,M04,site_A::M04
27,site_B,M04,site_B::M04
5,site_A,M05,site_A::M05
33,site_B,M05,site_B::M05
28,site_B,M06,site_B::M06
25,site_B,M07,site_B::M07
26,site_B,M08,site_B::M08


### Traitement du piège A

Certains identifiants comme `M04` et `M05` sont présents dans les deux sites,
mais ils ne correspondent pas au même équipement.

Une clé composite `machine_key` a été créée en combinant `source` et
`machine_id`. Elle permet d'identifier chaque machine sans ambiguïté.

In [36]:
# Piège B — même colonne temp_c, mais l'échelle diffère selon la source.
fusion.groupby('source')['temp_c'].describe()[['mean', 'min', 'max']]
# TODO — une des deux sources est en °F (unité oubliée). Laquelle ?
# Sans la colonne `source`, cette anomalie était invisible.

,mean,min,max
source,,,
site_A,68.544000,64.2,76.6
site_B,153.681818,142.9,166.2


# Conservation de la valeur originale pour la traçabilité.

In [37]:
fusion["temp_originale"] = fusion["temp_c"]

fusion["unite_temperature_originale"] = fusion["source"].map(
    {
        "site_A": "C",
        "site_B": "F",
    }
)

# Conversion du site B de Fahrenheit vers Celsius.
mask_site_b = fusion["source"] == "site_B"

fusion.loc[mask_site_b, "temp_c"] = (
    fusion.loc[mask_site_b, "temp_c"] - 32
) * 5 / 9

print("Températures après harmonisation en Celsius :")

display(
    fusion.groupby("source")["temp_c"]
    .describe()[["mean", "min", "max"]]
)

Températures après harmonisation en Celsius :


,mean,min,max
source,,,
site_A,68.54400,64.200000,76.600000
site_B,67.60101,61.611111,74.555556


### Traitement du piège B

Les températures du site B étaient exprimées en Fahrenheit malgré le nom
de colonne `temp_c`.

Elles ont été converties en Celsius. La valeur originale et son unité sont
conservées afin de garantir la traçabilité de la transformation.

In [38]:
# Piège C — debit_l_min n'existe que pour un site -> NaN pour l'autre.
fusion.groupby('source')['debit_l_min'].apply(lambda s: s.isna().mean())
# TODO — colonne réellement absente, ou juste pas transmise ? -> question client.

source
site_A    0.0
site_B    1.0
Name: debit_l_min, dtype: float64

# On n'invente pas de valeur de débit pour le site B.

In [39]:
fusion["debit_disponible"] = fusion["debit_l_min"].notna()

print("Disponibilité du débit par source :")

display(
    fusion.groupby("source")["debit_disponible"]
    .value_counts()
)

Disponibilité du débit par source :


source  debit_disponible
site_A  True                25
site_B  False               22
Name: count, dtype: int64

### Traitement du piège C

La colonne `debit_l_min` est présente uniquement dans l'export du site A.

Les valeurs manquantes du site B sont conservées : aucune valeur de débit
n'est inventée. La colonne `debit_disponible` permet de savoir si cette mesure
était présente dans la source.

Une question reste à poser au client : le site B ne mesure-t-il pas le débit,
ou cette colonne a-t-elle simplement été oubliée dans l'export ?

In [40]:
print("Dimensions finales :", fusion.shape)

print("\nNombre de lignes par source :")
display(fusion["source"].value_counts())

print("\nAperçu final :")
display(fusion.head())
display(fusion.tail())

assert len(fusion) == len(a) + len(b)
assert fusion["source"].notna().all()
assert fusion["machine_key"].notna().all()
assert set(fusion["source"]) == {"site_A", "site_B"}

print("Validation OK")
print("- 47 lignes conservées")
print("- provenance conservée")
print("- identifiants machines désambiguïsés")
print("- températures harmonisées en Celsius")
print("- valeurs absentes non inventées")

Dimensions finales : (47, 11)

Nombre de lignes par source :


source
site_A    25
site_B    22
Name: count, dtype: int64


Aperçu final :


,ts,machine_id,temp_c,vibration_mm_s,debit_l_min,source,firmware,machine_key,temp_originale,unite_temperature_originale,debit_disponible
0,2026-06-01T06:00:00,M01,72.5,1.77,124.7,site_A,NaN,site_A::M01,72.5,C,True
1,2026-06-01T07:00:00,M04,69.9,2.36,125.7,site_A,NaN,site_A::M04,69.9,C,True
2,2026-06-01T08:00:00,M04,64.6,2.40,126.3,site_A,NaN,site_A::M04,64.6,C,True
3,2026-06-01T09:00:00,M03,69.5,2.32,117.2,site_A,NaN,site_A::M03,69.5,C,True
4,2026-06-01T10:00:00,M03,64.2,1.83,116.3,site_A,NaN,site_A::M03,64.2,C,True


,ts,machine_id,temp_c,vibration_mm_s,debit_l_min,source,firmware,machine_key,temp_originale,unite_temperature_originale,debit_disponible
42,2026-06-01T23:00:00,M05,69.222222,3.30,NaN,site_B,v1.2,site_B::M05,156.6,F,False
43,2026-06-02T00:00:00,M04,68.444444,3.76,NaN,site_B,v1.3,site_B::M04,155.2,F,False
44,2026-06-02T01:00:00,M05,74.555556,2.51,NaN,site_B,v1.2,site_B::M05,166.2,F,False
45,2026-06-02T02:00:00,M08,67.388889,1.81,NaN,site_B,v1.2,site_B::M08,153.3,F,False
46,2026-06-02T03:00:00,M05,64.333333,1.23,NaN,site_B,v1.3,site_B::M05,147.8,F,False


Validation OK
- 47 lignes conservées
- provenance conservée
- identifiants machines désambiguïsés
- températures harmonisées en Celsius
- valeurs absentes non inventées


### Conclusion

Les exports des sites A et B ont été réunis avec une colonne `source`
conservant la provenance de chaque ligne.

Trois écarts ont été identifiés et traités :

1. des identifiants machines identiques entre les deux sites ;
2. une différence d'unité de température ;
3. une colonne de débit absente du site B.

La colonne `source` est conservée afin d'assurer la traçabilité, de résoudre
les conflits d'identifiants et de permettre des contrôles ultérieurs par site.